# 05. RT-DETR (query-based)

Ultralytics RT-DETR-L з вагами COCO за тим самим бюджетом, що й YOLO у `04_yolo`: 960 px, 20 епох, ті самі train / `val_sub.txt` / оцінка.

| Запуск | Що змінено | Тип |
|---|---|---|
| `baseline` | 300 запитів (queries) | — |
| `exp1_queries500` | 300 → 500 запитів: у кадрі до ~180 об'єктів, а в декодер потрапляють лише top-k ознак енкодера | гіперпараметр |
| `exp2_giou` | вага GIoU у втраті й у вартості bipartite matching 2 → 4, поверх кращого з попередніх | функція втрат |

In [ ]:
EXPERIMENT = "baseline"
SMOKE_TEST = False

In [ ]:
EXPERIMENTS = {
    "baseline": {"queries": 300, "giou_gain": 2.0},
    "exp1_queries500": {"queries": 500, "giou_gain": 2.0},
    "exp2_giou": {"queries": "best", "giou_gain": 4.0},
}

IMGSZ = 960
BATCH = 2
EPOCHS = 20
LR = 1e-4
SEED = 42
WORKERS = 16

## 1. Шляхи і дані

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import ultralytics
from ultralytics import RTDETR

ON_KAGGLE = Path("/kaggle/input").exists()
if ON_KAGGLE:
    for name in ("hw2_eval.py", "hw2_ultra.py"):
        sys.path.append(str(next(Path("/kaggle/input").glob(f"**/{name}")).parent))
import hw2_eval as E
import hw2_ultra as U

DATA, PREPARED, RUNS = U.paths()
SETUP = dict(EXPERIMENTS[EXPERIMENT])
if SETUP["queries"] == "best":
    SETUP["queries"] = EXPERIMENTS[U.best_run(RUNS / "rtdetr", ["baseline", "exp1_queries500"])]["queries"]
OUT = RUNS / "rtdetr" / (f"smoke_{EXPERIMENT}" if SMOKE_TEST else EXPERIMENT)
print(U.gpu_name(), SETUP, OUT, sep="\n")

In [ ]:
val = U.val_frames(DATA)
test = U.test_frames(DATA)
val_sub = val[val["pos"] % U.VAL_STEP == 0]
if SMOKE_TEST:
    EPOCHS = 1
    val, test, val_sub = val.head(4), test.head(4), val_sub.head(8)
data_yaml = U.write_data_yaml(PREPARED, list(val_sub["path"]), OUT)
pd.Series({"val (кожна епоха)": len(val_sub), "val (повний)": len(val), "test": len(test)})

## 2. Модель і втрата

Кількість запитів задається аргументом `nq` у `RTDETRDecoder`; беремо конфіг `rtdetr-l.yaml` з Ultralytics і змінюємо лише цей аргумент. Запити в RT-DETR не мають власних навчених векторів (вони вибираються з ознак енкодера), тому ваги COCO підходять для будь-якого `nq`.

Втрата: Ultralytics `RTDETRDetectionLoss` має ваги `class 1, bbox 5, giou 2` і такі самі ваги вартості в `HungarianMatcher` (`class 2, bbox 5, giou 2`). L1 рахується в нормованих координатах, тож для дрібних рамок вона майже нульова; GIoU не залежить від масштабу. Для експ. 2 вагу GIoU піднімаємо до 4 і у втраті, і у вартості зіставлення.

In [ ]:
from ultralytics.models.utils.loss import RTDETRDetectionLoss
from ultralytics.nn.tasks import RTDETRDetectionModel


def init_criterion(self):
    gain = {"class": 1, "bbox": 5, "giou": SETUP["giou_gain"], "no_object": 0.1, "mask": 1, "dice": 1}
    criterion = RTDETRDetectionLoss(nc=self.nc, loss_gain=gain, use_vfl=True)
    criterion.matcher.cost_gain["giou"] = SETUP["giou_gain"]
    return criterion


RTDETRDetectionModel.init_criterion = init_criterion

base_cfg = Path(ultralytics.__file__).parent / "cfg" / "models" / "rt-detr" / "rtdetr-l.yaml"
cfg_text = base_cfg.read_text().replace("RTDETRDecoder, [nc]]", f"RTDETRDecoder, [nc, 256, {SETUP['queries']}]]")
assert f"[nc, 256, {SETUP['queries']}]" in cfg_text
OUT.mkdir(parents=True, exist_ok=True)
cfg_path = OUT / "rtdetr-l.yaml"
cfg_path.write_text(cfg_text)

## 3. Навчання

AdamW, lr 1e-4, weight decay 1e-4 (як у статті RT-DETR), без підвищеного lr зсувів під час розігріву; batch 2, бо batch 4 при 960 px не вміщається у 8 ГБ GPU (Ultralytics накопичує градієнти до номінального batch 64, тож крок оптимізатора той самий); решта — стандартний рецепт Ultralytics, як у YOLO. Перерваний запуск продовжується з `last.pt`.

In [ ]:
elapsed = U.timer()
if U.resumable(OUT):
    RTDETR(OUT / "weights" / "last.pt").train(resume=True)
elif not (OUT / "weights" / "last.pt").exists():
    RTDETR(str(cfg_path)).load("rtdetr-l.pt").train(
        data=str(data_yaml), imgsz=IMGSZ, batch=BATCH, epochs=EPOCHS, seed=SEED, deterministic=True,
        optimizer="AdamW", lr0=LR, weight_decay=1e-4, warmup_bias_lr=0.0, workers=WORKERS, cache="disk",
        project=str(OUT.parent), name=OUT.name, exist_ok=True, fraction=0.002 if SMOKE_TEST else 1.0,
        plots=True, verbose=False)
train_minutes = elapsed()
history = U.history(OUT)
history[["epoch", "mAP", "AP50"]].round(4).tail(5)

## 4. Повний val, швидкість, сабміт

RT-DETR не використовує NMS: беремо до 300 рамок з найвищими оцінками.

In [ ]:
best_model = RTDETR(OUT / "weights" / "best.pt")
config = {"experiment": EXPERIMENT, **SETUP, "gpu": U.gpu_name(), "imgsz": IMGSZ, "batch": BATCH, "epochs": EPOCHS,
          "lr": LR, "seed": SEED, "train_min": train_minutes}
summary, per_class, clips, speed = U.finish(best_model, OUT, config, IMGSZ, val, test, DATA)
print(f"Навчання: {train_minutes:.0f} хв, швидкість: {speed:.1f} мс/кадр")
display(summary.round(4), per_class.round(4), clips.round(4))

## Джерела

**Бібліотеки**
- Ultralytics 8.3.40 — RT-DETR-L, ваги COCO, навчання: https://docs.ultralytics.com/models/rtdetr/
- faster-coco-eval через спільний `hw2_eval.py`.

**Код, взятий або адаптований**
- Ваги втрати й вартості зіставлення за замовчуванням — `ultralytics/models/utils/loss.py::DETRLoss`, `ultralytics/models/utils/ops.py::HungarianMatcher`; `init_criterion` — копія `ultralytics/nn/tasks.py::RTDETRDetectionModel.init_criterion` зі зміненими вагами.
- Аргумент `nq` (кількість запитів) — `ultralytics/nn/modules/head.py::RTDETRDecoder`.

**Статті**
- DETR (bipartite matching, без NMS) — arXiv:2005.12872; Deformable DETR — arXiv:2010.04159; DN-DETR (denoising) — arXiv:2203.01305; RT-DETR — arXiv:2304.08069 (AdamW, lr 1e-4, wd 1e-4).
- GIoU — arXiv:1902.09630.

**Наші рішення:** бюджет як у YOLO (960 px, 20 епох), 500 запитів через щільність сцен з EDA, вага GIoU 4 і у втраті, і у зіставленні.